## Imports

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from argparse import Namespace
from sklearn.utils import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from torchmetrics import Accuracy, F1Score

## Loading Dataset

In [2]:
df = pd.read_csv('./../data/processed/20newsgroup_preprocessed_own.csv', on_bad_lines='skip', delimiter=";")
print(df.shape)
df.head()

(18828, 3)


,target,text,text_cleaned
0,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismresources altatheismarchive...
1,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: A...,archivename atheismintroduction altatheismarch...
2,alt.atheism,From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...,article charley wingate writes well john quite...
3,alt.atheism,From: mathew <mathew@mantis.co.uk>\nSubject: R...,kings become philosophers philosophers become ...
4,alt.atheism,From: strom@Watson.Ibm.Com (Rob Strom)\nSubjec...,article pidaorg pidaorg bob mcgwier writes how...


In [3]:
df = df.dropna(subset=['text_cleaned'])
print(df.shape)

(18792, 3)


## Prepare BERT

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # fast and decent quality
X_bert = model.encode(df['text_cleaned'].tolist(), show_progress_bar=True)
print(X_bert.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/588 [00:00<?, ?it/s]

(18792, 384)
[[ 1.25528295e-02 -4.10585627e-02 -5.11533730e-02  6.39109313e-03
   8.84484351e-02  1.01927863e-02 -8.46678391e-02 -3.89942974e-02
   3.25095579e-02  4.10558246e-02 -5.26613789e-03 -4.93914634e-02
   2.13530590e-03 -1.11915972e-02 -7.73480954e-03  5.99906892e-02
  -1.17426226e-02  1.96864512e-02  5.23252785e-03 -1.71134155e-02
  -3.48222479e-02  1.04444526e-01 -2.15549092e-03  2.18526670e-03
  -2.21658424e-02 -3.73451747e-02 -9.62080806e-03 -4.25627157e-02
  -4.88256961e-02 -1.35161933e-02 -5.01156300e-02  3.01888119e-03
   3.81938368e-02 -2.59652417e-02  4.35805842e-02 -1.74001344e-02
   6.28648922e-02  8.70952979e-02  9.21684727e-02 -6.13418967e-02
  -4.53861989e-02 -3.55406925e-02 -7.29432926e-02 -5.45754619e-02
  -3.55321243e-02 -6.32485189e-03 -7.66701102e-02 -3.69550362e-02
   5.96297393e-03 -6.72317147e-02 -8.38710219e-02 -3.15111242e-02
   2.60466170e-02  4.68659997e-02 -9.24396217e-02  2.49064006e-02
  -8.65282118e-03 -1.50543116e-02 -4.69682403e-02 -1.07651435e-

## Tokeniying the documents

In [116]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=60000, stop_words="english", ngram_range = (1,2))

# Transform text data into TF-IDF features
X_tfidf = vectorizer.fit_transform(df['text_cleaned'])
X_tfidf = X_tfidf.toarray()

# Show shape of transformed data
print("TF-IDF Matrix Shape:", X_tfidf.shape)

TF-IDF Matrix Shape: (18792, 60000)


## Encode target labels

In [5]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['target'])
print("Unique labels in df:", np.unique(df['target']))
print("Unique labels in full dataset:", np.unique(y_encoded))

Unique labels in df: ['alt.atheism' 'comp.graphics' 'comp.os.ms-windows.misc'
 'comp.sys.ibm.pc.hardware' 'comp.sys.mac.hardware' 'comp.windows.x'
 'misc.forsale' 'rec.autos' 'rec.motorcycles' 'rec.sport.baseball'
 'rec.sport.hockey' 'sci.crypt' 'sci.electronics' 'sci.med' 'sci.space'
 'soc.religion.christian' 'talk.politics.guns' 'talk.politics.mideast'
 'talk.politics.misc' 'talk.religion.misc']
Unique labels in full dataset: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


## Create dataset and dataloader

In [6]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_bert, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

## Create a simple NN

In [7]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.main = nn.Sequential(
            nn.Linear(cfg.n_in, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(cfg.n_hidden, cfg.n_hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(cfg.n_hidden, cfg.n_out)  # raw logits for CrossEntropyLoss
        )

    def forward(self, x):
        return self.main(x)

## Model training

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)
if torch.cuda.is_available():
    print("GPU Name: ", torch.cuda.get_device_name())

cfg = Namespace(
    n_in = X_bert.shape[1], 
    n_hidden = 256, 
    n_out = 20,
    epochs = 10,
)
model = MultilayerPerceptron(cfg).to(device)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define metrics
acc_metric = Accuracy(task="multiclass", num_classes=20, average='macro').to(device)
f1_metric = F1Score(task="multiclass", num_classes=20, average='macro').to(device)

# Training loop
for epoch in range(cfg.epochs):
    model.train()
    running_loss = 0.0

    acc_metric.reset()
    f1_metric.reset()
    
    for X_batch, y_batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{cfg.epochs}"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)
        preds = torch.softmax(outputs, dim=1)  # probabilities

        # Compute loss
        loss = criterion(outputs, y_batch)

        # Backward + optimizer step
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

        # Compute accuracy and F1 Score
        pred_labels = torch.argmax(preds, dim=1)  # Predicted class for each sample
        true_labels = y_batch  # True labels

        acc_metric.update(pred_labels.detach(), true_labels)
        f1_metric.update(pred_labels.detach(), true_labels)

    acc = acc_metric.compute().item()
    f1 = f1_metric.compute().item()

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}, Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")

Using device: cpu


Training Epoch 1/10:   0%|          | 0/3759 [00:00<?, ?it/s]

Epoch 1, Loss: 1.3555, Accuracy: 0.5165, F1 Score: 0.5110


Training Epoch 2/10:   0%|          | 0/3759 [00:00<?, ?it/s]

Epoch 2, Loss: 0.8623, Accuracy: 0.7159, F1 Score: 0.7123


...

Epoch 8, Loss: 0.3371, Accuracy: 0.8965, F1 Score: 0.8965


Training Epoch 9/10:   0%|          | 0/3759 [00:00<?, ?it/s]

Epoch 9, Loss: 0.3013, Accuracy: 0.9053, F1 Score: 0.9052


Training Epoch 10/10:   0%|          | 0/3759 [00:00<?, ?it/s]

Epoch 10, Loss: 0.2680, Accuracy: 0.9182, F1 Score: 0.9183


## Evaluation

In [14]:
model.eval()

total = 0
correct = 0

for texts, labels in tqdm(test_loader, desc="Testing"):
    texts = texts.to(device)
    labels = labels.to(device)

    outputs = model(texts)  # [B, C]
    preds = torch.argmax(outputs, dim=1)  # [B]

    correct += (preds == labels).sum().item()
    total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/940 [00:00<?, ?it/s]

Test Accuracy: 80.34%
